# SLAM-ASR: Türkçe Konuşma Tanıma (t1-sa: Spectral Attention)
### ISSAI Turkish Speech Corpus (~218 Saat) & Spektral Dikkat (Spectral Attention) Geliştirmesi

> **Referans Makale:** *An Embarrassingly Simple Approach for LLM with Strong ASR Capacity* ([arXiv:2402.08846](https://arxiv.org/abs/2402.08846))
> **Veri Seti:** ISSAI Turkish Speech Corpus (ISSAI_TSC_218) ([Hugging Face](https://huggingface.co/datasets/issai/Turkish_Speech_Corpus))
> **Deney Dalı (Branch):** `t1-sa` (Spectral Attention Projector)

Bu notebook, **t1-sa** deneyinde geliştirilen **Spectral Attention (Cross-Subband Spectral Self-Attention & Channel Gating)** destekli projektör mimarisi ile SLAM-ASR modelini Google Colab üzerinde eğitmek ve test etmek için hazırlanmıştır.

---
### t1-sa Mimari Yapısı:
* **Akustik Kodlayıcı:** `openai/whisper-large-v3` - **Dondurulmuş (Frozen)**
* **Dil Modeli (LLM):** `Qwen/Qwen2.5-7B-Instruct` - **Dondurulmuş (Frozen)**
* **Projektör (`t1-sa: SpectralAttentionProjector`):**
  1. **Cross-Subband Spectral Self-Attention (16 Alt-Bant):** Akustik özellik boyutunu (1280) 16 spektral alt-banda (her biri 80 boyutlu) böler. Bu alt-bantlar arasında çok başlıklı self-attention kurarak temel frekans (F0), üst armonikler ve ünlü formantları (F1-F2) arasındaki harmonik bağıntıları frekans kanalları boyunca modeller.
  2. **Dynamic Spectral Channel Gating (Squeeze-and-Excitation):** Zamansal havuzlama ve darboğaz katmanları ile frekans kanallarını dinamik olarak yeniden kalibre eder; belirgin formant frekanslarını öne çıkarırken arka plan ortam gürültüsü kanallarını baskılar.
  3. **Zamansal Sıkıştırma (Downsampling):** Spektral olarak arındırılmış k=5 ardışık çerçevenin birleştirilmesi (50 Hz -> 10 Hz).
  4. **Lineer Projeksiyon:** Birleştirilmiş özellikleri LLM gömme boyutuna (3584) eşleme.


## 1. GPU Kontrolü, Google Drive ve API Anahtarları Yetkilendirmesi
Colab oturumunuzun GPU hızlandırıcıya bağlı olduğunu doğrulayın. Eğitimin ve veri indirmelerinin kesintisiz ilerlemesi, model ağırlıklarının otomatik yedeklenmesi için **Google Drive**, **HF_TOKEN** ve **MDC_API_KEY** izinlerini en baştan verin (Menü: `Runtime` -> `Change runtime type` -> `T4 GPU` veya `A100 GPU`).


In [1]:
# GPU kullanılabilirliğini kontrol edin
!nvidia-smi

import torch
print(f"[*] PyTorch Versiyonu: {torch.__version__}")
print(f"[*] CUDA Kullanılabilir mi?: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"[*] Aktif GPU: {torch.cuda.get_device_name(0)}")
    print(f"[*] VRAM: {round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)} GB")

# Google Drive'ı bağlayın (Eğitim bittiğinde yedeklemenin kesintisiz yapılabilmesi için izni baştan verin)
try:
    from google.colab import drive
    drive.mount("/content/drive")
    print("[*] Google Drive başarıyla bağlandı.")
except Exception as e:
    print(f"[!] Google Drive bağlantı notu: {e}")

# 3. Colab Secrets (🔑) Yetkilendirmeleri (HF_TOKEN & MDC_API_KEY)
# Model indirmelerinde hız limitine takılmamak ve veri indirmelerinin kesintisiz sürmesi için izinleri baştan verin.
import os
try:
    from google.colab import userdata

    # Hugging Face Token (Hızlı indirme ve model erişimi için)
    try:
        hf_token = userdata.get("HF_TOKEN")
        if hf_token:
            os.environ["HF_TOKEN"] = hf_token
            os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token
            print("[*] HF_TOKEN başarıyla alındı ve ortam değişkenlerine eklendi.")
    except Exception:
        print("[i] HF_TOKEN Colab Secrets'ta tanımlı değil veya izin verilmedi (İsteğe bağlı).")

    # Mozilla Data Collective API Key (Common Voice veri seti indirmesi için)
    try:
        mdc_api_key = userdata.get("MDC_API_KEY")
        if mdc_api_key:
            os.environ["MDC_API_KEY"] = mdc_api_key
            print("[*] MDC_API_KEY başarıyla alındı ve ortam değişkenlerine eklendi.")
    except Exception:
        print("[i] MDC_API_KEY Colab Secrets'ta tanımlı değil veya izin verilmedi (İsteğe bağlı).")
except ImportError:
    pass


Fri Oct 16 05:00:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   29C    P0             48W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 2. Depo Dosyalarını Hazırlama ve Bağımlılıkları Yükleme
GitHub reposunu çekin veya çalışma dizinine geçin ve gerekli paketleri yükleyin.

In [2]:
import os

# Repoyu klonlayın ve çalışma dizinine geçin
if not os.path.exists("slam_asr"):
    if not os.path.exists("slam-asr-reimplement"):
        !git clone https://github.com/gokhanersoy/slam-asr-reimplement.git
    %cd slam-asr-reimplement/tr/tsc/t1-sa

# Bağımlılıkları yükleyin
!pip install -q -r requirements.txt
!pip install -q huggingface_hub soundfile torchaudio bitsandbytes accelerate


Cloning into 'slam-asr-reimplement'...
remote: Enumerating objects: 1007, done.
remote: Counting objects: 100% (1007/1007), done.
remote: Compressing objects: 100% (828/828), done.
remote: Total 1007 (delta 269), reused 900 (delta 162), pack-reused 0 (from 0)
Receiving objects: 100% (1007/1007), 6.74 MiB | 12.31 MiB/s, done.
Resolving deltas: 100% (269/269), done.
/content/slam-asr-reimplement/tr/tsc
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 73.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 149.3 MB/s eta 0:00:00


## 3. Kod Dosyalarının Doğrulanması ve Birim Testler
Türkçe karakter normalizasyonu (`İ -> i`, `I -> ı`), projektör katmanı, dinamik batchleme ve donanım profilleme testlerini koşturun.

In [ ]:
!python run_tests.py

test_variable_length_waveform_collation (test_collator.TestCollator.test_variable_length_waveform_collation) ... ok
test_whisper_mel_collation (test_collator.TestCollator.test_whisper_mel_collation) ... ok
test_a100_40gb_tier_recommendations (test_device_utils.TestDeviceUtils.test_a100_40gb_tier_recommendations) ... ok
test_cpu_tier_recommendations (test_device_utils.TestDeviceUtils.test_cpu_tier_recommendations) ... ok
test_get_device_specs (test_device_utils.TestDeviceUtils.test_get_device_specs) ... ok
test_h100_tier_recommendations (test_device_utils.TestDeviceUtils.test_h100_tier_recommendations) ... ok
test_l4_tier_recommendations (test_device_utils.TestDeviceUtils.test_l4_tier_recommendations) ... ok
test_mps_tier_recommendations (test_device_utils.TestDeviceUtils.test_mps_tier_recommendations) ... ok
test_resolve_auto_hyperparameters (test_device_utils.TestDeviceUtils.test_resolve_auto_hyperparameters) ... ok
test_rtx_a4000_tier_recommendations (test_device_utils.TestDeviceUtil

## 4. Hızlı Doğrulama: Sentetik Türkçe Veri ile Test Eğitimi (30 Saniye)
21 GB'lık büyük veri setini indirmeden önce, tüm eğitim döngüsünün, gradyan akışının ve model ağırlıklarının hatasız çalıştığını sentetik verimizle test edin.

In [ ]:
# 1. Sentetik Türkçe test verisini üretin
!python scripts/create_toy_turkish_dataset.py

# 2. Hızlı doğrulama eğitimini koşturun (50 adım)
!python train.py --config config/train_toy_debug_turkish.yaml

[*] Generating synthetic Turkish toy dataset...
[*] Created 8 train samples in: /content/slam-asr-reimplement/tr/tsc/data/sample_train_tr.jsonl
[*] Created 2 val samples in:   /content/slam-asr-reimplement/tr/tsc/data/sample_val_tr.jsonl
[*] Turkish toy dataset generated successfully.
[*] HARDWARE-ADAPTIVE HYPERPARAMETER PROFILER
    Device Detected:      NVIDIA RTX PRO 6000 Blackwell Server Edition (94.97 GB VRAM), Compute sm_120
    Hardware Profile:     Ultra-High VRAM (H100 / A100 80GB)
    Profile Strategy:     Full 16-bit precision (Paper Setup). Maximum throughput without quantization.
----------------------------------------------------------------------
    Active Settings:
      • torch_dtype:       float32
      • load_in_4bit:      False (NF4)
      • batch_size:        2
      • grad_accumulation: 1  --> Effective Batch: 2
      • mixed_precision:   bf16=False, fp16=False
      • num_workers:       0
[*] Initializing SLAM-ASR with config: config/train_toy_debug_turkish.yam

## 5. Gerçek ISSAI_TSC_218 Türkçe Ses Korpusu Hazırlığı
Hugging Face Hub üzerinden `issai/Turkish_Speech_Corpus` veri seti çekilir (`ISSAI_TSC_218.tar.gz`).
Transkriptler Türkçe özel karakter duyarlılığıyla işlenir ve `train.jsonl`, `val.jsonl`, `test.jsonl` manifestoları oluşturulur.

> **İpucu:** İlk denemeyi daha hızlı yapmak isterseniz `--max_samples 1000` veya `--max_samples 5000` parametresi ekleyebilirsiniz.

In [ ]:
# ISSAI Turkish Speech Corpus'u indirin ve hazırlayın
!python scripts/prepare_issai_tsc.py --download --output_dir data/issai_tsc

[*] Downloading ISSAI_TSC_218.tar.gz from huggingface.co/datasets/issai/Turkish_Speech_Corpus...

ISSAI_TSC_218.tar.gz: downloading bytes:   1% 190M/21.4G [00:01<01:20, 265MB/s, 14.6MB/s  ]
ISSAI_TSC_218.tar.gz: downloading bytes:   3% 536M/21.4G [00:02<00:49, 425MB/s, 43.9MB/s  ]
ISSAI_TSC_218.tar.gz: downloading bytes:   3% 588M/21.4G [00:02<00:45, 453MB/s, 48.0MB/s  ]
ISSAI_TSC_218.tar.gz: reconstructing file:   2% 469M/21.4G [00:02<01:36, 217MB/s, 37.2MB/s  ]
ISSAI_TSC_218.tar.gz: downloading bytes:   4% 867M/21.4G [00:03<00:37, 541MB/s, 70.7MB/s  ]
ISSAI_TSC_218.tar.gz: downloading bytes:   5% 1.08G/21.4G [00:03<00:36, 557MB/s, 87.1MB/s  ]
ISSAI_TSC_218.tar.gz: downloading bytes:   5% 1.16G/21.4G [00:03<00:32, 613MB/s, 92.4MB/s  ]
ISSAI_TSC_218.tar.gz: downloading bytes:   6% 1.37G/21.4G [00:03<00:29, 668MB/s,  110MB/s  ]
ISSAI_TSC_218.tar.gz: downloading bytes:   7% 1.44G/21.4G [00:03<00:29, 671MB/s,  115MB/s  ]
ISSAI_TSC_218.tar.gz: downloading bytes:   8% 1.68G/21.4G [00:04<00:

## 6. Donanıma Göre Otomatik Optimize Olan Türkçe SLAM-ASR Eğitimi
Donanım profilleyicimiz (`device_utils`) GPU'nuzun VRAM'ine ve mimarisine göre en verimli ayarları otomatik belirler:
* **Colab T4 (16 GB) / RTX A4000 (16 GB):** 4-bit NF4 kuantizasyon + bfloat16/float16 + batch 4 + gradient accumulation 4 $\to$ Efektif Batch 16.
* **A100 (40/80 GB) / L4 (24 GB):** Yüksek throughput ve tam hassasiyet.

In [ ]:
# Donanıma göre otomatik optimize edilen Türkçe SLAM-ASR eğitimi
!python train.py --config config/hardware_adaptive_turkish.yaml

[*] HARDWARE-ADAPTIVE HYPERPARAMETER PROFILER
    Device Detected:      NVIDIA RTX PRO 6000 Blackwell Server Edition (94.97 GB VRAM), Compute sm_120
    Hardware Profile:     Ultra-High VRAM (H100 / A100 80GB)
    Profile Strategy:     Full 16-bit precision (Paper Setup). Maximum throughput without quantization.
----------------------------------------------------------------------
    Active Settings:
      • torch_dtype:       bfloat16
      • load_in_4bit:      False (NF4)
      • batch_size:        16
      • grad_accumulation: 1  --> Effective Batch: 16
      • mixed_precision:   bf16=True, fp16=False
      • num_workers:       4
[*] Initializing SLAM-ASR with config: config/hardware_adaptive_turkish.yaml
config.json: 100% 1.27k/1.27k [00:00<00:00, 7.44MB/s]

model.safetensors: downloading bytes:   5% 146M/3.09G [00:00<00:11, 263MB/s, 11.2MB/s  ]
model.safetensors: reconstructing file:   2% 67.1M/3.09G [00:00<00:40, 73.8MB/s]
model.safetensors: downloading bytes:   8% 256M/3.09G [

## 7. Model Değerlendirmesi (Türkçe WER ve CER Hesaplama)
Eğitilen modelin test seti üzerindeki performansını ölçün. Türkçe büyük/küçük harf ve noktalama normalizasyonu otomatik uygulanır.

In [ ]:
!python evaluate.py \
    --config config/hardware_adaptive_turkish.yaml \
    --checkpoint outputs/slam_asr_turkish_adaptive/best_checkpoint \
    --manifest data/issai_tsc/test.jsonl \
    --language tr \
    --beam_size 4

[*] Loading model on cuda...
Loading weights: 100% 1259/1259 [00:00<00:00, 14388.32it/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 339/339 [00:00<00:00, 15018.10it/s]
[*] Loading projector weights from: outputs/slam_asr_turkish_adaptive/best_checkpoint/projector.pt
[*] Evaluating 3484 samples with beam_size=4...
Evaluating:   0% 0/218 [00:00<?, ?it/s]/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:1455: UserWarning: Passing `repetition_penalty` with `inputs_embeds` and without `input_ids` to `generate` will apply the penalty only to newly generated tokens, not to the prompt.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:1463: UserWarning: Passing `no_repeat_ngram_size` with `inputs_embeds` and without `input_ids` to `generate` will apply n-gram constraints only to newly generated tokens, not to the prompt.
  warnings.warn(
Evaluating: 100% 218/218 [06:19<00:00,  1.74s/it]



## 8. İnteraktif Çıkarım (Ses Dinleme ve Transkripsiyon)
Bir Türkçe ses kaydını dinleyin ve modelinizin konuşmayı nasıl yazıya döktüğünü canlı olarak görün.

In [ ]:
import json
import IPython.display as ipd

# Test setinden ilk örneği bulun
sample_wav = None
sample_text = ""

manifest_file = "data/issai_tsc/test.jsonl" if os.path.exists("data/issai_tsc/test.jsonl") else "data/sample_val_tr.jsonl"
if os.path.exists(manifest_file):
    with open(manifest_file, "r", encoding="utf-8") as f:
        first_line = f.readline()
        if first_line:
            data = json.loads(first_line)
            sample_wav = data.get("audio_filepath")
            sample_text = data.get("text", "")

if sample_wav and os.path.exists(sample_wav):
    print(f"[*] Seçilen Ses: {sample_wav}")
    print(f"[*] Gerçek Metin (Ground Truth): {sample_text}\n")
    display(ipd.Audio(sample_wav))

    !python inference.py \
        --audio "{sample_wav}" \
        --config config/hardware_adaptive_turkish.yaml \
        --checkpoint outputs/slam_asr_turkish_adaptive/best_checkpoint/projector.pt \
        --prompt "Konuşmayı metne dönüştürün."
else:
    print("[!] Test ses dosyası bulunamadı. Lütfen ses dosyasının yolunu elle belirtin.")

[*] Seçilen Ses: /content/slam-asr-reimplement/tr/tsc/data/issai_tsc/extracted/ISSAI_TSC_218/Test/3476.wav
[*] Gerçek Metin (Ground Truth): kestikleri için iş aramıyorlar iş aramadıkları zaman da



[*] Loading SLAM-ASR on cuda...
Loading weights: 100% 1259/1259 [00:00<00:00, 14217.33it/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 339/339 [00:00<00:00, 23041.14it/s]
[*] Loading projector weights from: outputs/slam_asr_turkish_adaptive/best_checkpoint/projector.pt
[*] Processing audio: /content/slam-asr-reimplement/tr/tsc/data/issai_tsc/extracted/ISSAI_TSC_218/Test/3476.wav
[*] Transcribing (beam_size=4)...
/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:1455: UserWarning: Passing `repetition_penalty` with `inputs_embeds` and without `input_ids` to `generate` will apply the penalty only to newly generated tokens, not to the prompt.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:1463: UserWarning: Passing `no_repeat_ngram_size` with `inputs_embeds` and without `input_ids` to `generate` will apply n-gram constraints only to newly generated tokens, not to the prompt.
  warn

## 9. Zaman Damgalı Google Drive Yedeklemesi
Eğitilen ağırlıkları (`best_checkpoint`), test sonuçlarını (`eval_results.jsonl`), manifestoları ve konfigürasyonu Google Drive'a tarih/saat damgalı olarak kalıcı yedekleyin.


In [ ]:
# 9. Adım: Tüm Model Ağırlıklarını, Manifestoları ve Test Sonuçlarını Drive'a Yedekleme
import os
from datetime import datetime
from google.colab import drive

# 1. Google Drive bağlantısını kontrol edin (Henüz bağlı değilse bağlayın)
if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

# 2. Zaman damgalı hedef klasörü belirleyin
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
drive_target_dir = f"/content/drive/MyDrive/checkpoints/{timestamp}_slam_asr_tr_tsc_t1_sa"
os.makedirs(drive_target_dir, exist_ok=True)

# 3. Model Ağırlıklarını Kopyalayın
!cp -r outputs/slam_asr_turkish_adaptive/best_checkpoint "{drive_target_dir}/"
if os.path.exists("outputs/slam_asr_turkish_adaptive/final_checkpoint"):
    !cp -r outputs/slam_asr_turkish_adaptive/final_checkpoint "{drive_target_dir}/"

# 4. (Varsa) Değerlendirme Çıktısını Kopyalayın
if os.path.exists("eval_results.jsonl"):
    !cp eval_results.jsonl "{drive_target_dir}/"
if os.path.exists("eval_results_cv_generalization.jsonl"):
    !cp eval_results_cv_generalization.jsonl "{drive_target_dir}/"

# 5. Veri Manifestolarını Kopyalayın (train/val/test jsonl)
if os.path.exists("data/issai_tsc"):
    os.makedirs(f"{drive_target_dir}/manifests", exist_ok=True)
    !cp data/issai_tsc/*.jsonl "{drive_target_dir}/manifests/"
elif os.path.exists("data/sample_train_tr.jsonl"):
    os.makedirs(f"{drive_target_dir}/manifests", exist_ok=True)
    !cp data/sample_*.jsonl "{drive_target_dir}/manifests/"

# 6. Kullanılan Config Dosyasını Kopyalayın
!cp config/hardware_adaptive_turkish.yaml "{drive_target_dir}/"

print("\n" + "=" * 65)
print(f"[*] Tam Yedekleme Başarıyla Tamamlandı!")
print(f"[*] Hedef Dizin: {drive_target_dir}")
print("    • best_checkpoint & final_checkpoint")
print("    • eval_results.jsonl (Test tahminleri ve WER analizi)")
print("    • train/val/test manifestoları")
print("    • hardware_adaptive_turkish.yaml konfigürasyonu")
print("=" * 65)



### 10. Çapraz Korpus Testi: Mozilla Common Voice (CV-TR) ile Değerlendirme (Out-of-Domain Generalization)
Makalede (Bölüm 4.5 ve Tablo 6) açıklandığı üzere, **ISSAI_TSC_218** ile eğitilen modelin eğitimde hiç görmediği **Mozilla Common Voice Türkçe (CV-TR)** test kümesi (11.869 kayıt) üzerindeki alan dışı (out-of-domain) genelleme başarımı test edilir.

> **Makale Bulgusu:** Common Voice konuşmaları kontrollü ortamda temiz okunan cümlelerden oluştuğu için, TSC ile eğitilen modeller Common Voice test setinde in-domain ISSAI_TSC sonucuna göre daha düşük (daha iyi) hata oranları elde etmektedir (ör. Whisper-Large-v3 için WER %14.89 $\to$ %9.3, CER %3.96 $\to$ %2.0).


In [3]:
# 10. Adım: TSC ile Eğitilen Modelin Mozilla Common Voice (CV-TR) Test Seti Üzerinde Değerlendirilmesi
import os
import glob
import json
import re

# 1. Drive'ı Bağlayın (Henüz bağlı değilse)
if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

# 2. Model checkpoint'ini belirleyin (Önce yerel, sonra Drive)
checkpoint_path = "outputs/slam_asr_turkish_adaptive/best_checkpoint"
if not os.path.exists(checkpoint_path):
    candidates = sorted(glob.glob("/content/drive/MyDrive/checkpoints/*slam_asr_tr_tsc*/best_checkpoint"), key=lambda p: (re.search(r"(\d{8}_\d{6})", p).group(1) if re.search(r"(\d{8}_\d{6})", p) else "", p))
    if candidates:
        checkpoint_path = candidates[-1]

# 3. Common Voice test manifestosunu bulun ve ses dosyalarının diskte var olduğunu doğrulayın
def is_valid_manifest(path):
    if not path or not os.path.exists(path):
        return False
    api_key = os.environ.get("MDC_API_KEY")
    if not api_key:
    api_key = os.environ.get("MDC_API_KEY")
    if not api_key:
        try:
            from google.colab import userdata
            api_key = userdata.get("MDC_API_KEY")
        except Exception:
            api_key = None
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue
                data = json.loads(line)
                audio_path = data.get("audio_filepath", "")
                return os.path.exists(audio_path)
        return False
    return False

cv_manifest = None
manifest_candidates = [
    "data/cv/test.jsonl",
    "../../cv/t0/data/cv/test.jsonl",
    "../cv/data/cv/test.jsonl",
    "/content/slam-asr-reimplement/tr/cv/t0/data/cv/test.jsonl",
    "/content/slam-asr-reimplement/tr/cv/data/cv/test.jsonl",
]
manifest_candidates.extend(sorted(glob.glob("/content/drive/MyDrive/checkpoints/*slam_asr_tr_cv*/manifests/test.jsonl")))

for p in manifest_candidates:
    if is_valid_manifest(p):
        cv_manifest = p
        break

# 4. CV ses dosyaları diskte yoksa ve MDC_API_KEY tanımlıysa otomatik indir/hazırla
if not cv_manifest:
    api_key = os.environ.get("MDC_API_KEY")
    if not api_key:
    api_key = os.environ.get("MDC_API_KEY")
    if not api_key:
        try:
            from google.colab import userdata
            api_key = userdata.get("MDC_API_KEY")
        except Exception:
            api_key = None

    prep_script = "scripts/prepare_common_voice.py"
    if not os.path.exists(prep_script):
        for cand in ["../../cv/t0/scripts/prepare_common_voice.py", "../cv/scripts/prepare_common_voice.py", "/content/slam-asr-reimplement/tr/cv/t0/scripts/prepare_common_voice.py", "/content/slam-asr-reimplement/tr/cv/scripts/prepare_common_voice.py"]:
            if os.path.exists(cand):
                prep_script = cand
                break
    if os.path.exists(prep_script) and api_key:
        print("[*] Common Voice test verisi yerel diskte bulunamadı. Mozilla Data Collective üzerinden hazırlanıyor...")
        !python "{prep_script}" --download --api_key "{api_key}" --output_dir data/cv
        if is_valid_manifest("data/cv/test.jsonl"):
            cv_manifest = "data/cv/test.jsonl"

# 5. Değerlendirmeyi çalıştır ve sonucu Drive'a kaydet
if cv_manifest and os.path.exists(checkpoint_path):
    print(f"[*] Kullanılan Checkpoint: {checkpoint_path}")
    print(f"[*] Common Voice Test Manifestosu: {cv_manifest}\n")
    !python evaluate.py \
        --config config/hardware_adaptive_turkish.yaml \
        --checkpoint "{checkpoint_path}" \
        --manifest "{cv_manifest}" \
        --output_file eval_results_cv_generalization.jsonl \
        --language tr \
        --beam_size 4

    # Değerlendirme çıktısını Drive'daki checkpoint klasörüne yedekle
    drive_target = None
    if "/content/drive/MyDrive" in checkpoint_path:
        drive_target = os.path.dirname(checkpoint_path.rstrip("/"))
    elif "drive_target_dir" in globals() and os.path.exists(drive_target_dir):
        drive_target = drive_target_dir

    if drive_target and os.path.exists(drive_target) and os.path.exists("eval_results_cv_generalization.jsonl"):
        !cp eval_results_cv_generalization.jsonl "{drive_target}/"
        print(f"\n[*] CV değerlendirme sonuçları Drive'a yedeklendi: {drive_target}/eval_results_cv_generalization.jsonl")
else:
    if not os.path.exists(checkpoint_path):
        print(f"[!] Model checkpoint'i ({checkpoint_path}) bulunamadı. Lütfen önce 6. adımı çalıştırıp modeli eğitin veya Drive'daki bir checkpoint'i kontrol edin.")
    if not cv_manifest:
        print("[!] Common Voice test ses dosyaları yerel diskte bulunamadı.")
        print("    (Drive'da test.jsonl manifestosu bulunsa dahi, içindeki .mp3 ses dosyaları mevcut Colab diskinde yok.)")
        print("    İpucu: CV verisini otomatik indirmek için Colab Secrets'a (🔑) 'MDC_API_KEY' ekleyip bu hücreyi tekrar çalıştırabilirsiniz.")


Mounted at /content/drive
[*] Common Voice test verisi yerel diskte bulunamadı. Mozilla Data Collective üzerinden hazırlanıyor...
[*] Connecting to Mozilla Data Collective REST API...
[*] Requesting presigned download URL from Mozilla Data Collective API...
    URL: https://mozilladatacollective.com/api/datasets/cmu5wkah500c2o10719lem8m8/download
[*] Downloading dataset to: /content/slam-asr-reimplement/tr/tsc/data/cv/common-voice-scripted-speech-27-0-turkis-b14760fe.tar.gz
    Progress: 0% (8.0 MB / 2853.8 MB)
    Progress: 5% (144.0 MB / 2853.8 MB)
    Progress: 10% (288.0 MB / 2853.8 MB)
    Progress: 15% (432.0 MB / 2853.8 MB)
    Progress: 20% (576.0 MB / 2853.8 MB)
    Progress: 25% (720.0 MB / 2853.8 MB)
    Progress: 30% (864.0 MB / 2853.8 MB)
    Progress: 35% (1000.0 MB / 2853.8 MB)
    Progress: 40% (1144.0 MB / 2853.8 MB)
    Progress: 45% (1288.0 MB / 2853.8 MB)
    Progress: 50% (1432.0 MB / 2853.8 MB)
    Progress: 55% (1576.0 MB / 2853.8 MB)
    Progress: 60% (1720.0 MB

## 11. Modeli Belleğe Yükle ve Canlı Mikrofon Kaydı ile Konuşmayı Metne Dönüştür
Model GPU belleğine bir kere yüklenir ve hazır bekler. Ardından tarayıcınızdan mikrofona Türkçe konuşursunuz ve model konuşmanızı 1 saniye içinde yazıya döker.


In [ ]:
# 11. Adım: Modeli Belleğe Yükle ve Mikrofon ile Canlı Konuşmayı Metne Dönüştür
import os
import glob
import subprocess
import yaml
from base64 import b64decode
import torch
import IPython.display as ipd
from google.colab import output

from slam_asr.models.slam_model import SLAM_ASR, SLAMASRConfig
from slam_asr.data.audio_processor import AudioProcessor
from slam_asr.utils.device_utils import resolve_auto_hyperparameters

# 1. Drive'ı Bağla ve En Güncel Checkpoint'i Bul
if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

checkpoint_candidates = sorted(glob.glob("/content/drive/MyDrive/checkpoints/*slam_asr_tr_tsc*/best_checkpoint/projector.pt"), key=lambda p: (re.search(r"(\d{8}_\d{6})", p).group(1) if re.search(r"(\d{8}_\d{6})", p) else "", p))

if checkpoint_candidates:
    best_ckpt = checkpoint_candidates[-1]
elif os.path.exists("outputs/slam_asr_turkish_adaptive/best_checkpoint/projector.pt"):
    best_ckpt = "outputs/slam_asr_turkish_adaptive/best_checkpoint/projector.pt"
else:
    raise FileNotFoundError("Hiçbir best_checkpoint/projector.pt dosyası bulunamadı!")

# 2. Modeli GPU'ya Yükle (Zaten Yüklüyse Tekrar Yüklemez!)
if "loaded_slam_model" not in globals():
    print(f"[*] Model checkpoint'i kullanılıyor:\n    {best_ckpt}\n")
    print("[*] 1/3: Model GPU VRAM'ine yükleniyor... Lütfen bekleyin...")

    cfg = yaml.safe_load(open("config/hardware_adaptive_turkish.yaml"))
    model_cfg = cfg.get("model", {})
    data_cfg = cfg.get("data", {})
    model_cfg, data_cfg, _ = resolve_auto_hyperparameters(model_cfg, data_cfg, {})

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model_config = SLAMASRConfig(**model_cfg)

    model = SLAM_ASR(config=model_config)
    if not getattr(model_config, "load_in_4bit", False) and not getattr(model_config, "load_in_8bit", False):
        model = model.to(device)
    else:
        model.speech_encoder = model.speech_encoder.to(device)
        model.projector = model.projector.to(device)

    state_dict = torch.load(best_ckpt, map_location=device)
    model.projector.load_state_dict(state_dict)
    model.eval()

    audio_proc = AudioProcessor(
        sampling_rate=data_cfg.get("sampling_rate", 16000),
        is_whisper=data_cfg.get("is_whisper", True),
        whisper_mel_bins=data_cfg.get("whisper_mel_bins", 128),
    )

    loaded_slam_model = model
    loaded_audio_proc = audio_proc
    loaded_device = device
    loaded_model_config = model_config
    print("[*] 2/3: Model başarıyla VRAM'e yüklendi ve hazır bekliyor! 🚀\n")
else:
    print("[*] Model zaten VRAM'de hazır bekliyor (Yeniden yükleme yapılmadı). 🚀\n")

# 3. Model Hazır Olduktan Sonra Mikrofon Butonunu Göster
RECORD_JS = """
const sleep = time => new Promise(resolve => setTimeout(resolve, time));
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader();
  reader.onloadend = e => resolve(e.target.result);
  reader.readAsDataURL(blob);
});
var recordAudio = () => new Promise(async (resolve, reject) => {
  try {
    const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
    const recorder = new MediaRecorder(stream);
    const chunks = [];
    recorder.ondataavailable = e => chunks.push(e.data);

    const container = document.createElement("div");
    container.style.cssText = "padding: 12px; margin: 10px 0; background: #1e1e2e; border-radius: 8px; color: white; font-family: sans-serif; display: inline-block;";

    const btn = document.createElement("button");
    btn.innerHTML = "🔴 Kaydı Başlat (Konuşmaya Başlayın)";
    btn.style.cssText = "padding: 12px 24px; font-size: 16px; font-weight: bold; background: #e74c3c; color: white; border: none; border-radius: 6px; cursor: pointer; transition: 0.3s; box-shadow: 0 4px 6px rgba(0,0,0,0.3);";
    container.appendChild(btn);
    document.body.appendChild(container);

    let isRecording = false;

    btn.onclick = () => {
      if (!isRecording) {
        recorder.start();
        isRecording = true;
        btn.innerHTML = "⏹️ Kaydı Bitir (Sonucu Gör)";
        btn.style.background = "#27ae60";
      } else {
        recorder.stop();
        btn.innerHTML = "⏳ Model transkribe ediyor...";
        btn.disabled = true;
        btn.style.background = "#7f8c8d";
      }
    };

    recorder.onstop = async () => {
      const blob = new Blob(chunks, { type: "audio/webm" });
      const base64Text = await b2text(blob);
      container.remove();
      resolve(base64Text);
    };
  } catch (err) {
    reject(err);
  }
});
recordAudio();
"""

print("[*] 3/3: Mikrofon hazır. Konuşmak için aşağıdaki butona basın:")
audio_data = output.eval_js(RECORD_JS)

# 4. Sesi 16 kHz Mono WAV Formatına Dönüştür
webm_file = "live_temp.webm"
wav_file = "live_mic_recording.wav"

with open(webm_file, "wb") as f:
    f.write(b64decode(audio_data.split(",")[1]))

subprocess.run([
    "ffmpeg", "-y", "-i", webm_file,
    "-ar", "16000", "-ac", "1",
    wav_file
], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print("\n[*] Kaydettiğiniz Ses:")
display(ipd.Audio(wav_file))

# 5. Bellekteki Model ile Anında Çıkarım
speech_tensor = loaded_audio_proc.process(wav_file).unsqueeze(0).to(loaded_device)
use_amp = (loaded_model_config.torch_dtype in ("float16", "bfloat16")) and loaded_device.type == "cuda"
amp_dtype = torch.bfloat16 if loaded_model_config.torch_dtype == "bfloat16" else torch.float16

with torch.no_grad():
    if use_amp:
        autocast_ctx = (
            torch.amp.autocast("cuda", dtype=amp_dtype)
            if hasattr(torch, "amp") and hasattr(torch.amp, "autocast")
            else torch.cuda.amp.autocast(dtype=amp_dtype)
        )
        with autocast_ctx:
            transcription = loaded_slam_model.generate(
                speech_inputs=speech_tensor,
                prompts=["Konuşmayı metne dönüştürün."],
                num_beams=4,
            )[0]
    else:
        transcription = loaded_slam_model.generate(
            speech_inputs=speech_tensor,
            prompts=["Konuşmayı metne dönüştürün."],
            num_beams=4,
        )[0]

print("\n" + "=" * 55)
print(f"🗣️  Sizin Konuşmanız: {transcription}")
print("=" * 55)
